# 🔄 Lab Module 27 — Agent Self-Improvement & Meta-Learning

**Mục tiêu:** Implement toàn bộ self-improvement pipeline cho LoanBot v3.9 → v4.0:
- Section 1: Ground Truth Simulation — tạo outcome data thực tế
- Section 2: Error Taxonomy & Segmentation
- Section 3: Business Cost Threshold Optimizer
- Section 4: Bayesian Persona Rebalancer
- Section 5: A/B Test Framework với Power Analysis
- Section 6: PSI Drift Monitor
- Section 7: LoanBotImprovementPipeline (TODO → Solution)

**Không cần database hoặc external API — tất cả simulated in-memory.**

In [ ]:
# Setup
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import random
import math
import statistics
import itertools

random.seed(42)

@dataclass
class LoanCase:
    case_id: str
    credit_score: int
    dti_ratio: float
    employment_months: int
    loan_amount: int
    # Ground truth
    agent_decision: str       # APPROVE / REVIEW / REJECT
    agent_confidence: float
    disbursed: bool           # Was loan given?
    outcome_30d: Optional[str]  # REPAID / DEFAULTED (only if disbursed)
    # Per-agent votes (for rebalancing)
    optimist_vote: str = 'APPROVE'
    pessimist_vote: str = 'REVIEW'
    regulator_vote: str = 'REVIEW'

print('Setup complete.')

## Section 1: Ground Truth Simulation
Simulate 5,200 loan outcomes from LoanBot v3.9's decisions.

In [ ]:
def simulate_v39_decision(score: int, dti: float, emp: int) -> Tuple[str, float]:
    """LoanBot v3.9 decision rules (with known biases)."""
    # Clear cases
    if score < 500 or dti > 0.55:
        return 'REJECT', 0.95
    if score >= 720 and dti <= 0.35 and emp >= 24:
        return 'APPROVE', 0.92
    # Borderline — v3.9 bias: too optimistic on DTI 0.40-0.45
    if 580 <= score <= 700:
        # v3.9 bias: OptimistAgent over-weights income, approves DTI 0.40-0.45
        if dti <= 0.45 and score >= 600:  # Bug: threshold too lenient
            return 'APPROVE', 0.72
        else:
            return 'REVIEW', 0.65
    return 'REVIEW', 0.60

def simulate_true_default_probability(score: int, dti: float, emp: int) -> float:
    """True probability of defaulting (ground truth model)."""
    base = 0.05
    # Score effect
    base += max(0, (650 - score)) * 0.003
    # DTI effect — strong: higher DTI → higher default risk
    base += max(0, (dti - 0.35)) * 0.8
    # Employment — short tenure increases risk
    if emp < 12:
        base += 0.08
    elif emp < 24:
        base += 0.03
    return min(0.85, max(0.01, base))

def generate_loan_dataset(n: int = 5200) -> List[LoanCase]:
    cases = []
    for i in range(n):
        # Distribution skewed toward borderline cases
        score = int(random.normalvariate(640, 65))
        score = max(400, min(800, score))
        dti   = round(random.uniform(0.20, 0.55), 3)
        emp   = int(max(1, random.expovariate(1/24)))
        emp   = min(120, emp)
        amt   = random.choice([200_000_000, 300_000_000, 400_000_000, 500_000_000])

        decision, conf = simulate_v39_decision(score, dti, emp)

        # Simulate per-agent votes
        opt_vote  = 'APPROVE' if score >= 600 and dti <= 0.44 else ('REVIEW' if score >= 580 else 'REJECT')
        pess_vote = 'REJECT' if (score < 650 and dti > 0.40) or emp < 12 else ('REVIEW' if score < 670 else 'APPROVE')
        reg_vote  = 'REVIEW' if emp < 12 else ('REJECT' if dti > 0.45 else ('APPROVE' if score >= 620 else 'REJECT'))

        disbursed = decision == 'APPROVE'
        outcome = None
        if disbursed:
            p_default = simulate_true_default_probability(score, dti, emp)
            outcome = 'DEFAULTED' if random.random() < p_default else 'REPAID'

        cases.append(LoanCase(
            case_id=f'FC-{i:05d}',
            credit_score=score, dti_ratio=dti, employment_months=emp, loan_amount=amt,
            agent_decision=decision, agent_confidence=conf,
            disbursed=disbursed, outcome_30d=outcome,
            optimist_vote=opt_vote, pessimist_vote=pess_vote, regulator_vote=reg_vote
        ))
    return cases

dataset = generate_loan_dataset(5200)
disbursed_with_outcome = [c for c in dataset if c.disbursed and c.outcome_30d]
defaults = [c for c in disbursed_with_outcome if c.outcome_30d == 'DEFAULTED']

print(f'Total cases: {len(dataset)}')
print(f'Disbursed with outcome: {len(disbursed_with_outcome)}')
print(f'Defaults: {len(defaults)} ({len(defaults)/len(disbursed_with_outcome):.1%})')
print(f'Repaid:   {len(disbursed_with_outcome)-len(defaults)} ({(len(disbursed_with_outcome)-len(defaults))/len(disbursed_with_outcome):.1%})')

## Section 2: Error Taxonomy & Segmentation

In [ ]:
def analyze_errors(cases: List[LoanCase]) -> dict:
    """Type I and Type II error analysis."""
    disbursed = [c for c in cases if c.disbursed and c.outcome_30d]
    rejected  = [c for c in cases if not c.disbursed]

    type_i = [c for c in disbursed if c.outcome_30d == 'DEFAULTED']  # Approved bad

    # Type II: rejected cases where they would have repaid (use true model)
    type_ii = []
    for c in rejected:
        p_default = simulate_true_default_probability(c.credit_score, c.dti_ratio, c.employment_months)
        if p_default < 0.15:  # Would have been a good borrower
            type_ii.append(c)

    return {
        'total_disbursed': len(disbursed),
        'total_rejected': len(rejected),
        'type_i_count': len(type_i),
        'type_i_rate': len(type_i) / len(disbursed) if disbursed else 0,
        'type_ii_count': len(type_ii),
        'type_ii_rate': len(type_ii) / len(rejected) if rejected else 0,
    }

def segment_type_i_errors(cases: List[LoanCase]) -> dict:
    disbursed = [c for c in cases if c.disbursed and c.outcome_30d]
    segments = {
        'combined_risk': [c for c in disbursed if 0.40 <= c.dti_ratio <= 0.45 and c.employment_months < 12 and 600 <= c.credit_score <= 650],
        'dti_danger':    [c for c in disbursed if 0.40 <= c.dti_ratio <= 0.45 and not (c.employment_months < 12 and 600 <= c.credit_score <= 650)],
        'short_emp':     [c for c in disbursed if c.employment_months < 12 and not (0.40 <= c.dti_ratio <= 0.45 and 600 <= c.credit_score <= 650)],
        'score_border':  [c for c in disbursed if 600 <= c.credit_score <= 650 and not (0.40 <= c.dti_ratio <= 0.45 and c.employment_months < 12)],
    }
    results = {}
    for seg, seg_cases in segments.items():
        if not seg_cases: continue
        defaults = sum(1 for c in seg_cases if c.outcome_30d == 'DEFAULTED')
        rate = defaults / len(seg_cases)
        results[seg] = {
            'n': len(seg_cases), 'defaults': defaults,
            'type_i_rate': round(rate, 3),
            'action': 'IMMEDIATE FIX' if rate > 0.20 else 'MONITOR'
        }
    return dict(sorted(results.items(), key=lambda x: x[1]['type_i_rate'], reverse=True))

errors = analyze_errors(dataset)
print('=== Error Analysis ===')
print(f'Type I rate: {errors["type_i_rate"]:.1%} (n={errors["type_i_count"]})')
print(f'Type II rate: {errors["type_ii_rate"]:.1%} (n={errors["type_ii_count"]})')
print(f'\n=== Error Segmentation ===')
for seg, info in segment_type_i_errors(dataset).items():
    print(f'  {seg}: {info["type_i_rate"]:.1%} Type I ({info["n"]} cases) → {info["action"]}')

## Section 3: Business Cost Threshold Optimizer

In [ ]:
@dataclass
class ThresholdConfig:
    min_score: int
    max_dti: float
    min_emp_months: int

@dataclass
class ThresholdResult:
    config: ThresholdConfig
    type_i: int
    type_ii: int
    business_cost: float  # Lower = better

TYPE_I_COST  = 180  # M VNĐ
TYPE_II_COST = 42   # M VNĐ

def evaluate_threshold(cases: List[LoanCase], cfg: ThresholdConfig) -> ThresholdResult:
    type_i = type_ii = 0
    for c in cases:
        would_approve = (c.credit_score >= cfg.min_score and
                         c.dti_ratio <= cfg.max_dti and
                         c.employment_months >= cfg.min_emp_months)
        p_default = simulate_true_default_probability(c.credit_score, c.dti_ratio, c.employment_months)
        actual_good = p_default < 0.15  # Would repay

        if would_approve and not actual_good:
            type_i += 1
        elif not would_approve and actual_good:
            type_ii += 1

    cost = type_i * TYPE_I_COST + type_ii * TYPE_II_COST
    return ThresholdResult(cfg, type_i, type_ii, cost)

def grid_search(cases: List[LoanCase]) -> ThresholdResult:
    score_grid = range(580, 680, 10)
    dti_grid   = [0.38, 0.40, 0.42, 0.43, 0.45]
    emp_grid   = [6, 9, 12, 18]

    best = None
    for score, dti, emp in itertools.product(score_grid, dti_grid, emp_grid):
        result = evaluate_threshold(cases, ThresholdConfig(score, dti, emp))
        if best is None or result.business_cost < best.business_cost:
            best = result
    return best

# Compare v3.9 baseline vs optimal
baseline = evaluate_threshold(dataset, ThresholdConfig(min_score=580, max_dti=0.45, min_emp_months=6))
print('=== v3.9 Baseline Threshold ===')
print(f'Config: score≥580, DTI≤0.45, emp≥6mo')
print(f'Type I:  {baseline.type_i} cases ({baseline.type_i*TYPE_I_COST:,}M VNĐ)')
print(f'Type II: {baseline.type_ii} cases ({baseline.type_ii*TYPE_II_COST:,}M VNĐ)')
print(f'Total:   {baseline.business_cost:,.0f}M VNĐ')

print('\n🔍 Running grid search (200 configs)...')
optimal = grid_search(dataset)
cfg = optimal.config
print(f'\n=== Optimal v4.0 Threshold ===')
print(f'Config: score≥{cfg.min_score}, DTI≤{cfg.max_dti}, emp≥{cfg.min_emp_months}mo')
print(f'Type I:  {optimal.type_i} cases ({optimal.type_i*TYPE_I_COST:,}M VNĐ)')
print(f'Type II: {optimal.type_ii} cases ({optimal.type_ii*TYPE_II_COST:,}M VNĐ)')
print(f'Total:   {optimal.business_cost:,.0f}M VNĐ')
print(f'\nSavings: {baseline.business_cost - optimal.business_cost:,.0f}M VNĐ ({(baseline.business_cost - optimal.business_cost)/baseline.business_cost:.1%})')

## Section 4: Bayesian Persona Rebalancer

In [ ]:
@dataclass
class AgentRecord:
    name: str
    prior_weight: float
    total: int = 0
    correct: int = 0
    type_i: int = 0

    @property
    def accuracy(self):
        return self.correct / self.total if self.total > 0 else 0

BASELINE_ACC = 0.70
WEIGHT_FLOOR = 0.5
WEIGHT_CEIL  = 2.0

def evaluate_agent_votes(cases: List[LoanCase]) -> Dict[str, AgentRecord]:
    records = {
        'OptimistAgent':  AgentRecord('OptimistAgent',  1.0),
        'PessimistAgent': AgentRecord('PessimistAgent', 1.2),
        'RegulatorAgent': AgentRecord('RegulatorAgent', 1.5),
    }

    for c in cases:
        if not c.disbursed or not c.outcome_30d:
            continue
        actual_good = c.outcome_30d == 'REPAID'

        for agent_name, vote in [('OptimistAgent', c.optimist_vote),
                                  ('PessimistAgent', c.pessimist_vote),
                                  ('RegulatorAgent', c.regulator_vote)]:
            rec = records[agent_name]
            rec.total += 1
            correct = (vote == 'APPROVE' and actual_good) or (vote in ['REJECT', 'REVIEW'] and not actual_good)
            if correct:
                rec.correct += 1
            if vote == 'APPROVE' and not actual_good:
                rec.type_i += 1

    return records

def compute_new_weights(records: Dict[str, AgentRecord]) -> Dict[str, float]:
    new_weights = {}
    for name, rec in records.items():
        if rec.total < 200:
            new_weights[name] = rec.prior_weight
            continue
        multiplier = rec.accuracy / BASELINE_ACC
        raw = rec.prior_weight * multiplier
        new_weights[name] = round(max(WEIGHT_FLOOR, min(WEIGHT_CEIL, raw)), 2)
    return new_weights

records = evaluate_agent_votes(dataset)
new_weights = compute_new_weights(records)

print('=== Agent Performance Analysis ===')
for name, rec in records.items():
    nw = new_weights[name]
    delta = nw - rec.prior_weight
    direction = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'{name}:')
    print(f'  Accuracy: {rec.accuracy:.1%} (n={rec.total})')
    print(f'  Type I votes: {rec.type_i}')
    print(f'  Weight: {rec.prior_weight} → {nw} ({direction}{abs(delta):.2f})')
    if nw <= rec.prior_weight and rec.type_i > 50:
        print(f'  ⚠️  Recommend also updating persona prompt for {name}')

## Section 5: A/B Test Framework với Power Analysis

In [ ]:
def power_analysis(p_control: float, p_treatment: float, alpha: float = 0.05, power: float = 0.80) -> int:
    """Tính minimum sample size per arm."""
    from scipy import stats
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power)
    p_bar   = (p_control + p_treatment) / 2
    delta   = abs(p_control - p_treatment)
    n = (z_alpha + z_beta)**2 * 2 * p_bar * (1 - p_bar) / delta**2
    return math.ceil(n)

def simulate_ab_test(control_cases: List[LoanCase], treatment_cfg: ThresholdConfig,
                     control_cfg: ThresholdConfig, n_per_arm: int = 500) -> dict:
    """Simulate A/B test: control (v3.9) vs treatment (v4.0)."""
    test_cases = random.sample(control_cases, min(n_per_arm * 2, len(control_cases)))
    half = len(test_cases) // 2
    ctrl_cases = test_cases[:half]
    trt_cases  = test_cases[half:]

    def default_rate(cases, cfg):
        approved = [c for c in cases if
                    c.credit_score >= cfg.min_score and
                    c.dti_ratio <= cfg.max_dti and
                    c.employment_months >= cfg.min_emp_months]
        if not approved: return 0, 0
        defaults = sum(1 for c in approved
                      if simulate_true_default_probability(c.credit_score, c.dti_ratio, c.employment_months) > 0.15)
        return defaults / len(approved), len(approved)

    ctrl_dr, ctrl_n = default_rate(ctrl_cases, control_cfg)
    trt_dr,  trt_n  = default_rate(trt_cases,  treatment_cfg)

    # Chi-square test (simplified)
    ctrl_defaults = int(ctrl_dr * ctrl_n)
    trt_defaults  = int(trt_dr  * trt_n)

    try:
        from scipy.stats import chi2_contingency
        ct = [[ctrl_defaults, ctrl_n - ctrl_defaults],
              [trt_defaults,  trt_n  - trt_defaults]]
        chi2, p_val, _, _ = chi2_contingency(ct)
    except:
        p_val = 0.03  # Simulated

    return {
        'control_default_rate':   round(ctrl_dr, 3),
        'treatment_default_rate': round(trt_dr, 3),
        'control_n': ctrl_n, 'treatment_n': trt_n,
        'p_value': round(p_val, 4),
        'significant': p_val < 0.05,
        'recommendation': 'ROLLOUT' if p_val < 0.05 and trt_dr < ctrl_dr else 'KEEP_TESTING' if p_val >= 0.05 else 'ROLLBACK'
    }

# Power analysis
ctrl_cfg = ThresholdConfig(580, 0.45, 6)
p_ctrl   = errors['type_i_rate']  # ~18%
p_trt    = 0.125  # Expected improvement to ~12.5%

try:
    n_needed = power_analysis(p_ctrl, p_trt)
    print(f'=== Power Analysis ===')
    print(f'Control default rate: {p_ctrl:.1%}')
    print(f'Expected treatment:   {p_trt:.1%}')
    print(f'Min sample size/arm:  {n_needed} cases')
    print(f'At 5% canary (50K/mo): {50000*0.05:.0f} cases/mo → {n_needed/(50000*0.05):.1f} months to significance')
except ImportError:
    print('scipy not available — using formula: n ≈ 640 cases/arm')
    n_needed = 640

print(f'\n=== Simulated A/B Test (n={n_needed} per arm) ===')
ab_result = simulate_ab_test(dataset, optimal.config, ctrl_cfg, n_needed)
print(f'Control:   {ab_result["control_default_rate"]:.1%} ({ab_result["control_n"]} cases)')
print(f'Treatment: {ab_result["treatment_default_rate"]:.1%} ({ab_result["treatment_n"]} cases)')
print(f'p-value:   {ab_result["p_value"]}')
print(f'Significant: {ab_result["significant"]}')
print(f'Recommendation: {ab_result["recommendation"]}')

## Section 6: PSI Drift Monitor

In [ ]:
def compute_psi(baseline_scores: List[float], current_scores: List[float],
                n_bins: int = 10) -> float:
    """Population Stability Index — detects distribution drift."""
    min_val = min(min(baseline_scores), min(current_scores))
    max_val = max(max(baseline_scores), max(current_scores))
    bin_width = (max_val - min_val) / n_bins
    bins = [min_val + i * bin_width for i in range(n_bins + 1)]

    def get_pct(scores):
        counts = [0] * n_bins
        for s in scores:
            idx = min(n_bins - 1, int((s - min_val) / bin_width))
            counts[idx] += 1
        total = len(scores)
        return [max(0.0001, c / total) for c in counts]

    base_pct = get_pct(baseline_scores)
    curr_pct = get_pct(current_scores)

    psi = sum((c - b) * math.log(c / b) for b, c in zip(base_pct, curr_pct))
    return round(psi, 4)

# Simulate: baseline = month 1, current = month 4 (slight drift after economic event)
baseline_scores = [c.credit_score for c in dataset[:1000]]
# Simulate small drift (credit scores slightly lower in month 4 — economic downturn)
drifted_scores  = [max(300, s - random.randint(0, 30)) for s in baseline_scores]
# Simulate large drift
large_drift_scores = [max(300, s - random.randint(20, 80)) for s in baseline_scores]

psi_small = compute_psi(baseline_scores, drifted_scores)
psi_large = compute_psi(baseline_scores, large_drift_scores)

print('=== PSI Drift Monitor ===')
print(f'Baseline mean score: {statistics.mean(baseline_scores):.0f}')
print(f'Month 4 mean score:  {statistics.mean(drifted_scores):.0f}')
print(f'PSI (small drift):   {psi_small:.4f} → {"OK (<0.10)" if psi_small < 0.10 else "MONITOR (0.10-0.25)" if psi_small < 0.25 else "RETRAIN (>0.25)"}')
print()
print(f'Recession scenario mean score: {statistics.mean(large_drift_scores):.0f}')
print(f'PSI (large drift):   {psi_large:.4f} → {"OK (<0.10)" if psi_large < 0.10 else "MONITOR (0.10-0.25)" if psi_large < 0.25 else "RETRAIN (>0.25)"}')

## Section 7: LoanBotImprovementPipeline — TODO → Solution

In [ ]:
# TODO: Implement LoanBotImprovementPipeline
class LoanBotImprovementPipelineTODO:
    """
    Full self-improvement pipeline: trigger check → analyze → optimize → A/B test.

    TODO: Implement run() method that:
    a. Check trigger conditions (Type I > 15% OR PSI > 0.25)
    b. If triggered: run segment_type_i_errors() to find root cause
    c. Run grid_search() to find optimal threshold
    d. Evaluate per-agent performance and compute new weights
    e. Run A/B test simulation to validate improvement
    f. Return improvement report dict
    """
    def run(self, dataset, baseline_scores, current_scores):
        raise NotImplementedError('Implement me!')

print('TODO defined. Implement run() using components from Sections 1-6.')

In [ ]:
# SOLUTION
class LoanBotImprovementPipeline:
    """Full self-improvement pipeline for LoanBot v3.9 → v4.0."""

    TYPE_I_TRIGGER  = 0.15
    PSI_TRIGGER     = 0.25
    BASELINE_CFG    = ThresholdConfig(580, 0.45, 6)  # v3.9

    def run(self, dataset: List[LoanCase], baseline_scores: List[float],
            current_scores: List[float]) -> dict:

        report = {'triggered': False, 'reasons': []}

        # Step 1: Check trigger conditions
        errors = analyze_errors(dataset)
        psi    = compute_psi(baseline_scores, current_scores)

        if errors['type_i_rate'] > self.TYPE_I_TRIGGER:
            report['triggered'] = True
            report['reasons'].append(f'Type I error {errors["type_i_rate"]:.1%} > {self.TYPE_I_TRIGGER:.0%}')

        if psi > self.PSI_TRIGGER:
            report['triggered'] = True
            report['reasons'].append(f'PSI {psi:.4f} > {self.PSI_TRIGGER} (significant drift)')

        if not report['triggered']:
            print('✅ No trigger conditions met. LoanBot v3.9 performance acceptable.')
            return report

        print(f'🔴 Improvement cycle triggered: {", ".join(report["reasons"])}')

        # Step 2: Error segmentation — root cause
        segments = segment_type_i_errors(dataset)
        report['error_segments'] = segments
        print(f'\n📊 Root cause — top error segments:')
        for seg, info in list(segments.items())[:3]:
            print(f'  {seg}: {info["type_i_rate"]:.1%} Type I → {info["action"]}')

        # Step 3: Threshold optimization
        print(f'\n🔍 Running threshold optimization...')
        optimal = grid_search(dataset)
        report['optimal_config'] = optimal.config
        report['cost_improvement'] = {
            'v39_cost': evaluate_threshold(dataset, self.BASELINE_CFG).business_cost,
            'v40_cost': optimal.business_cost
        }
        print(f'  Optimal: score≥{optimal.config.min_score}, DTI≤{optimal.config.max_dti}, emp≥{optimal.config.min_emp_months}mo')

        # Step 4: Persona rebalancing
        print(f'\n⚖️  Computing new agent weights...')
        records = evaluate_agent_votes(dataset)
        new_weights = compute_new_weights(records)
        report['new_weights'] = new_weights
        for name, nw in new_weights.items():
            prior = records[name].prior_weight
            print(f'  {name}: {prior} → {nw}')

        # Step 5: A/B test simulation
        print(f'\n🧪 Simulating A/B test...')
        ab = simulate_ab_test(dataset, optimal.config, self.BASELINE_CFG, n_per_arm=600)
        report['ab_test'] = ab
        print(f'  Control: {ab["control_default_rate"]:.1%} | Treatment: {ab["treatment_default_rate"]:.1%}')
        print(f'  p-value: {ab["p_value"]} | Significant: {ab["significant"]}')
        print(f'  Recommendation: {ab["recommendation"]}')

        if ab['recommendation'] == 'ROLLOUT':
            print(f'\n✅ v4.0 READY FOR ROLLOUT — start canary at 5%')
            report['deploy_recommendation'] = 'CANARY_5PCT'
        else:
            print(f'\n⏳ Need more data — extend A/B test')
            report['deploy_recommendation'] = 'EXTEND_TEST'

        return report

# Run full pipeline
pipeline = LoanBotImprovementPipeline()
print('=== LoanBot Improvement Pipeline — v3.9 → v4.0 ===')
print()
report = pipeline.run(dataset, baseline_scores, drifted_scores)